### Reading the CSV file from catalog


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.read.csv( "/Volumes/workspace/default/likki/C1 (1).csv",header="True", inferSchema=True )

df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023|blank|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|New York|03-15-2023|  400|       B|
|       107|  Trent|   india|10-04-2023|  350|       B|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|     blank|  450|       C|
+----------+-------+--------+----------+-----+--------+

root
 |-- CustomerID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- JoinDate: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Category: 

##Handle "blank"/"Blank" values

In [0]:
from pyspark.sql.functions import *
df=df.replace("blank","Blank",None)
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023|Blank|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|New York|03-15-2023|  400|       B|
|       107|  Trent|   india|10-04-2023|  350|       B|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|     Blank|  450|       C|
+----------+-------+--------+----------+-----+--------+



In [0]:
from pyspark.sql.functions import lower,upper

df = df.withColumn("Country", upper(col("Country")))
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   INDIA|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023|Blank|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|NEW YORK|03-15-2023|  400|       B|
|       107|  Trent|   INDIA|10-04-2023|  350|       B|
|       108|    Bob|   INDIA|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|     Blank|  450|       C|
+----------+-------+--------+----------+-----+--------+



In [0]:
df = df.dropDuplicates()
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       102|    Bob|   INDIA|01-05-2023|  150|       B|
|       108|    Bob|   INDIA|05-01-2023|  150|       B|
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       107|  Trent|   INDIA|10-04-2023|  350|       B|
|       110|  Peggy|      UK|     Blank|  450|       C|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|NEW YORK|03-15-2023|  400|       B|
|       103|Charlie|      UK|20-02-2023|Blank|       C|
+----------+-------+--------+----------+-----+--------+



### Fix Date Format 

In [0]:
df = df.withColumn("Sales", col("Sales").cast("int"))
df = df.withColumn("JoinDate",
    when(col("JoinDate") == "blank", None).otherwise(col("JoinDate"))
)
df.display()

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-5921538348200446>, line 5
      1 df = df.withColumn("Sales", col("Sales").cast("int"))
      2 df = df.withColumn("JoinDate",
      3     when(col("JoinDate") == "blank", None).otherwise(col("JoinDate"))
      4 )
----> 5 df.display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:74, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     70 def df_display(df, *args, **kwargs):
     71     """
     72     df.display() is an alias for display(df). Run help(display) for more information.
     73     """
---> 74     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_tab

###practice

In [0]:
data = [
    (1, "Ravi", "india", "15-01-2023", "250", "A"),
    (2, None, "USA", "01-05-2023", "150", "B"),
    (3, "Anita", "India", "blank", "300", "A"),
    (4, "John", "UK", "20-02-2023", "-100", "C"),
    (4, "John", "UK", "20-02-2023", "-100", "C")
]

columns = ["customer_id", "name", "country", "join_date", "sales", "category"]

df = spark.createDataFrame(data, columns)
df.show()

+-----------+-----+-------+----------+-----+--------+
|customer_id| name|country| join_date|sales|category|
+-----------+-----+-------+----------+-----+--------+
|          1| Ravi|  india|15-01-2023|  250|       A|
|          2| NULL|    USA|01-05-2023|  150|       B|
|          3|Anita|  India|     blank|  300|       A|
|          4| John|     UK|20-02-2023| -100|       C|
|          4| John|     UK|20-02-2023| -100|       C|
+-----------+-----+-------+----------+-----+--------+



In [0]:
df = df.dropna(subset=["name"])

print("after removing null names")
df.show()

after removing null names
+-----------+-----+-------+----------+-----+--------+
|customer_id| name|country| join_date|sales|category|
+-----------+-----+-------+----------+-----+--------+
|          1| Ravi|  india|15-01-2023|  250|       A|
|          3|Anita|  India|     blank|  300|       A|
|          4| John|     UK|20-02-2023| -100|       C|
|          4| John|     UK|20-02-2023| -100|       C|
+-----------+-----+-------+----------+-----+--------+



In [0]:
df = df.withColumn(
    "join_date",
    when(col("join_date") == "blank", None).otherwise(col("join_date"))
)

print("after fixing blank values")
df.show()

after fixing blank values
+-----------+-----+-------+----------+-----+--------+
|customer_id| name|country| join_date|sales|category|
+-----------+-----+-------+----------+-----+--------+
|          1| Ravi|  india|15-01-2023|  250|       A|
|          3|Anita|  India|      NULL|  300|       A|
|          4| John|     UK|20-02-2023| -100|       C|
|          4| John|     UK|20-02-2023| -100|       C|
+-----------+-----+-------+----------+-----+--------+



In [0]:
df = df.withColumn("sales", col("sales").cast("int"))

print("after type conversion")
df.printSchema()

after type conversion
root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- join_date: string (nullable = true)
 |-- sales: integer (nullable = true)
 |-- category: string (nullable = true)



In [0]:
df = df.dropDuplicates()

print("after removing duplicates")
df.show()

after removing duplicates
+-----------+-----+-------+----------+-----+--------+
|customer_id| name|country| join_date|sales|category|
+-----------+-----+-------+----------+-----+--------+
|          1| Ravi|  india|15-01-2023|  250|       A|
|          3|Anita|  India|      NULL|  300|       A|
|          4| John|     UK|20-02-2023| -100|       C|
+-----------+-----+-------+----------+-----+--------+



In [0]:
df = df.filter(col("sales") > 0)

print("after removing negative sales")
df.show()

after removing negative sales
+-----------+-----+-------+----------+-----+--------+
|customer_id| name|country| join_date|sales|category|
+-----------+-----+-------+----------+-----+--------+
|          1| Ravi|  india|15-01-2023|  250|       A|
|          3|Anita|  India|      NULL|  300|       A|
+-----------+-----+-------+----------+-----+--------+



In [0]:
df = df.withColumn("country", initcap(col("country")))

print("after standardizing country")
df.show()

after standardizing country
+-----------+-----+-------+----------+-----+--------+
|customer_id| name|country| join_date|sales|category|
+-----------+-----+-------+----------+-----+--------+
|          1| Ravi|  India|15-01-2023|  250|       A|
|          3|Anita|  India|      NULL|  300|       A|
+-----------+-----+-------+----------+-----+--------+



In [0]:
df = df.withColumn("join_date", to_date(col("join_date"), "dd-MM-yyyy"))

print("after date conversion")
df.show()

after date conversion
+-----------+-----+-------+----------+-----+--------+
|customer_id| name|country| join_date|sales|category|
+-----------+-----+-------+----------+-----+--------+
|          1| Ravi|  India|2023-01-15|  250|       A|
|          3|Anita|  India|      NULL|  300|       A|
+-----------+-----+-------+----------+-----+--------+

